# 07 - Cluster Scaling Behavior

Analyze pods vs throughput curves, autoscaling behavior, and HPA setpoints vs load.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

try:
    %store -r df
except:
    np.random.seed(42)
    n = 50000
    df = pd.DataFrame({
        "timestamp_utc_iso": pd.date_range("2025-01-01", periods=n, freq="10ms"),
        "latency_us": np.random.lognormal(6, 0.5, n).astype(int),
        "worker_id": np.random.randint(0, 8, n),
    })

if "timestamp" not in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp_utc_iso"])


In [ ]:
# Workers vs throughput analysis
if "worker_id" in df.columns:
    df["second"] = df["timestamp"].dt.floor("S")
    
    # Active workers per second
    workers_per_second = df.groupby("second")["worker_id"].nunique()
    throughput_per_second = df.groupby("second").size()
    
    # Merge for correlation
    scaling_df = pd.DataFrame({
        "workers": workers_per_second,
        "throughput": throughput_per_second,
    }).dropna()
    
    print("Scaling Analysis:")
    print(f"  Workers range: {scaling_df['workers'].min()} - {scaling_df['workers'].max()}")
    print(f"  Throughput range: {scaling_df['throughput'].min()} - {scaling_df['throughput'].max()}")
    
    # Correlation
    corr = scaling_df["workers"].corr(scaling_df["throughput"])
    print(f"  Correlation (workers, throughput): {corr:.3f}")
    
    # Efficiency: throughput per worker
    scaling_df["efficiency"] = scaling_df["throughput"] / scaling_df["workers"]
    print(f"  Mean efficiency: {scaling_df['efficiency'].mean():.1f} msg/s/worker")
    
    # Plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    time_s = (scaling_df.index - scaling_df.index.min()).total_seconds()
    
    # Workers over time
    ax1 = axes[0, 0]
    ax1.plot(time_s, scaling_df["workers"].values, linewidth=1.5)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Active Workers")
    ax1.set_title("Worker Count Over Time")
    
    # Throughput over time
    ax2 = axes[0, 1]
    ax2.plot(time_s, scaling_df["throughput"].values, linewidth=1)
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Throughput (msg/s)")
    ax2.set_title("Throughput Over Time")
    
    # Scatter: workers vs throughput
    ax3 = axes[1, 0]
    ax3.scatter(scaling_df["workers"], scaling_df["throughput"], alpha=0.5, s=20)
    ax3.set_xlabel("Active Workers")
    ax3.set_ylabel("Throughput (msg/s)")
    ax3.set_title(f"Throughput vs Workers (r={corr:.2f})")
    
    # Efficiency over time
    ax4 = axes[1, 1]
    ax4.plot(time_s, scaling_df["efficiency"].values, linewidth=1)
    ax4.axhline(scaling_df["efficiency"].mean(), color="red", linestyle="--", label=f"Mean: {scaling_df['efficiency'].mean():.0f}")
    ax4.set_xlabel("Time (s)")
    ax4.set_ylabel("Efficiency (msg/s/worker)")
    ax4.set_title("Worker Efficiency Over Time")
    ax4.legend()
    
    plt.tight_layout()
    plt.show()


# 07 - Cluster Scaling Behavior

Analyze cluster scaling behavior and HPA performance.

## Objectives
- Analyze pods vs throughput curves
- Study autoscaling behavior
- Evaluate HPA setpoints vs load
- Identify scaling bottlenecks


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPERIMENT_ID = "exp_2025_0101_001"
DATA_PATH = f"../data/{EXPERIMENT_ID}/merged/merged.parquet"

df = pd.read_parquet(DATA_PATH)
if "timestamp_utc_iso" in df.columns:
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc_iso"])

print(f"Loaded {len(df):,} records")
print(f"Workers: {df['worker_id'].nunique() if 'worker_id' in df.columns else 'N/A'}")


In [ ]:
# Analyze per-worker throughput
if "worker_id" in df.columns and "timestamp_utc" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Operations per worker
    worker_counts = df["worker_id"].value_counts().sort_index()
    axes[0].bar(worker_counts.index.astype(str), worker_counts.values, color="#2196F3")
    axes[0].axhline(worker_counts.mean(), color="red", linestyle="--", label=f"Mean: {worker_counts.mean():.0f}")
    axes[0].set_xlabel("Worker ID")
    axes[0].set_ylabel("Operations")
    axes[0].set_title("Operations per Worker")
    axes[0].legend()
    
    # Throughput by worker over time
    for wid in df["worker_id"].unique()[:5]:  # First 5 workers
        worker_df = df[df["worker_id"] == wid].set_index("timestamp_utc")
        tp = worker_df.resample("5S").size()
        axes[1].plot(tp.values, label=f"Worker {wid}", alpha=0.7)
    axes[1].set_xlabel("Time (5s intervals)")
    axes[1].set_ylabel("Throughput")
    axes[1].set_title("Per-Worker Throughput")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
